# 03 - Benchmarks

Compares LimiX-2, LightGBM, XGBoost, logistic regression and the heuristic scorecard on the public
and synthetic datasets with stratified cross-validation.

The protocol is deliberately conservative:

* the same folds for every model;
* preprocessing fitted inside each training fold;
* LimiX receives only the training fold as its in-context reference set;
* AUC, KS, PR-AUC, Brier and log loss are reported.

This notebook uses one repeat to keep the runtime short. The published tables use `repeats=2` and
are in [benchmarks/results/benchmark_summary.md](../benchmarks/results/benchmark_summary.md). Do not
read the numbers below as the published results.

The Polish and SBA benchmarks are omitted here for runtime; run the CLI for all datasets, including
the 5k SBA sample documented in `docs/datasets.md`.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from opencashflow.benchmark import run_benchmark
from opencashflow.data.loaders import list_datasets

pd.set_option("display.width", 160)

## 1. Dataset registry

In [2]:
list_datasets()

,name,title,rows,features,default_rate,license,doi
0,polish,Polish Companies Bankruptcy,"7,027 (1year) to 10,503 (3year)",64,3.9%-6.9% depending on horizon,CC BY 4.0,10.24432/C5F600
1,german,Statlog (German Credit Data),"1,000",20 attributes,30.0%,CC BY 4.0,10.24432/C5NC77
2,bangladesh_sme,Structured Data Suffices: SME Credit Risk in B...,"1,200",27 columns including text narratives,25.25%,"CC BY 4.0 (data), MIT (code)",10.5281/zenodo.18525278
3,sba,Should This Loan be Approved or Denied? (SBA N...,"899,164",21 approval-time features after leakage removal,17.56%,CC BY-SA 4.0,
4,synthetic,OpenCashFlow synthetic SME portfolio,configurable,151 cash-flow features,"configurable, 12% by default",Apache-2.0 (generated data),


## 2. Run the benchmark

This uses three datasets, 300 synthetic companies, 20 months of history, three folds, one repeat and one LimiX pipeline on CPU. Expect several minutes. If LimiX is unavailable, the harness skips it with a warning.

In [3]:
root = Path.cwd()
if not (root / "pyproject.toml").exists() and (root.parent / "pyproject.toml").exists():
    root = root.parent

report = run_benchmark(
    datasets=["synthetic", "german", "bangladesh_sme"],
    models=["heuristic", "logistic", "lightgbm", "xgboost", "limix"],
    cv=3,
    repeats=1,
    seed=42,
    synthetic_companies=300,
    synthetic_months=20,
    limix_pipelines=1,
    output_dir=root / "benchmarks" / "results" / "notebook",
    verbose=True,
)
report.summary.round(3)

[dataset] synthetic


  [model] heuristic
  [model] logistic


  [model] lightgbm


  [model] xgboost


  [model] limix


arch_version=2.0 -> inference.v2_0.predictor (inference.v2_0.predictor)
Mixed precision is not supported for CPU inference, so it has been automatically disabled
use limix v2.0


... (6 outputs left out)

  [model] xgboost


  [model] limix


arch_version=2.0 -> inference.v2_0.predictor (inference.v2_0.predictor)
Mixed precision is not supported for CPU inference, so it has been automatically disabled
use limix v2.0


,dataset,model,variant,split,n,n_features,default_rate,roc_auc,roc_auc_std,roc_auc_ci_low,roc_auc_ci_high,ks,pr_auc,brier,log_loss,fit_seconds,predict_seconds,n_runs
0,bangladesh_sme,lightgbm,all,stratified,1200,19,0.252,0.895,0.022,0.841,0.949,0.624,0.753,0.111,0.351,0.531,0.010,3
1,bangladesh_sme,limix,all,stratified,1200,19,0.252,0.928,0.028,0.859,0.998,0.716,0.831,0.092,0.292,0.047,11.696,3
2,bangladesh_sme,logistic,all,stratified,1200,19,0.252,0.844,0.011,0.817,0.871,0.533,0.694,0.128,0.404,0.041,0.010,3
3,bangladesh_sme,xgboost,all,stratified,1200,19,0.252,0.898,0.018,0.854,0.943,0.666,0.751,0.113,0.351,0.733,0.015,3
4,german,lightgbm,all,stratified,1000,20,0.300,0.768,0.037,0.676,0.859,0.430,0.601,0.175,0.538,0.426,0.009,3
5,german,limix,all,stratified,1000,20,0.300,0.804,0.032,0.725,0.884,0.484,0.643,0.159,0.483,0.039,9.979,3
6,german,logistic,all,stratified,1000,20,0.300,0.776,0.028,0.707,0.845,0.449,0.591,0.171,0.514,0.077,0.014,3
7,german,xgboost,all,stratified,1000,20,0.300,0.785,0.030,0.712,0.859,0.448,0.615,0.169,0.521,0.667,0.014,3
8,synthetic,heuristic,all,stratified,300,151,0.113,0.889,0.045,0.776,1.001,0.725,0.746,0.073,0.260,0.000,0.001,3
9,synthetic,lightgbm,all,stratified,300,151,0.113,0.847,0.066,0.684,1.011,0.732,0.755,0.051,0.235,0.161,0.016,3


## 3. Ranking by dataset

In [4]:
pivot = report.summary.pivot(index="dataset", columns="model", values="roc_auc").sort_index()
fig, ax = plt.subplots(figsize=(11, 4))
pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("ROC-AUC (3-fold mean)")
ax.set_title("Model comparison by dataset")
ax.set_ylim(0.5, 1.0)
ax.legend(loc="lower right", fontsize=8)
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()
pivot.round(3)

model,heuristic,lightgbm,limix,logistic,xgboost
dataset,,,,,
bangladesh_sme,NaN,0.895,0.928,0.844,0.898
german,NaN,0.768,0.804,0.776,0.785
synthetic,0.889,0.847,0.894,0.802,0.859


## 4. Calibration matters too

AUC only measures ranking. Brier score penalises confident wrong probabilities.

In [5]:
brier = report.summary.pivot(index="dataset", columns="model", values="brier").sort_index()
brier.round(3)

model,heuristic,lightgbm,limix,logistic,xgboost
dataset,,,,,
bangladesh_sme,NaN,0.111,0.092,0.128,0.113
german,NaN,0.175,0.159,0.171,0.169
synthetic,0.073,0.051,0.047,0.073,0.050


## 5. Dataset notes

The Bangladesh loader drops `missed_payment_last_12m` because it is an exact copy of the target. German and Polish are tabular accounting/application datasets, not transaction ledgers.

In [6]:
from opencashflow.data.loaders import load_dataset

for name in ["german", "bangladesh_sme"]:
    bundle = load_dataset(name)
    print(bundle.summary())
    print("  leakage columns dropped:", bundle.metadata.get("dropped_leakage_columns", []))
    print("  categorical features:", len(bundle.categorical_features))

german: 1,000 rows x 20 features, default rate 30.00%, source UCI Machine Learning Repository, licence CC BY 4.0
  leakage columns dropped: []
  categorical features: 13
bangladesh_sme: 1,200 rows x 19 features, default rate 25.25%, source Zenodo, licence CC BY 4.0 (data), MIT (code)
  leakage columns dropped: ['missed_payment_last_12m']
  categorical features: 9


## 6. Interpretation guardrails

* In this one-repeat run LimiX has the highest mean ROC-AUC on all three datasets: 0.928 on
  Bangladesh SME, 0.804 on German and 0.894 on synthetic.
* Each model has three fold scores, so the intervals are wide. LimiX's synthetic interval runs from 0.699 to 1.089, and an AUC cannot exceed 1, so read `roc_auc_std` and the fold scores instead.
* The heuristic scorecard appears only for synthetic. The harness skips it on German and Bangladesh
  because they have no cash-flow features.
* The synthetic portfolio is the only transaction-level benchmark and its labels are generated.
* LimiX ran with one pipeline on CPU. A 32-member GPU run is not comparable.
* Do not claim state of the art from this notebook. Use it to compare implementations, features and
  preprocessing under a shared protocol.

Saved outputs are in `benchmarks/results/notebook/`.